# 01_prepare_eda

## Что делает ноутбук

Этот ноутбук открывает исследовательский пайплайн в воспроизводимом виде:
он читает только сырые таблицы из `data/raw/`, выполняет компактную подготовку
данных и сохраняет все промежуточные артефакты в `artifacts/`.

## Что читает

- `data/raw/train.csv`
- `data/raw/test.csv`
- `data/raw/stores.csv`
- `data/raw/oil.csv`
- `data/raw/holidays_events.csv`
- `data/raw/transactions.csv`

## Что сохраняет

- `artifacts/datasets/train_processed.csv`
- `artifacts/datasets/test_processed.csv`
- `artifacts/datasets/zero_sales.csv`
- `artifacts/datasets/top_pairs.csv`
- `artifacts/metadata/splits.json`

## Принцип воспроизводимости

- ноутбук не опирается на заранее подготовленные `train_processed/test_processed`;
- все downstream-ноутбуки должны читать только артефакты из `artifacts/`;
- путь от raw-данных до сплитов и списков серий здесь полностью детерминирован.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..')
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'
METADATA_DIR = ARTIFACTS_DIR / 'metadata'

for path in [DATASETS_DIR, METADATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(RAW_DATA_DIR / 'train.csv', parse_dates=['date'])
test_df = pd.read_csv(RAW_DATA_DIR / 'test.csv', parse_dates=['date'])
stores_df = pd.read_csv(RAW_DATA_DIR / 'stores.csv')
oil_df = pd.read_csv(RAW_DATA_DIR / 'oil.csv', parse_dates=['date'])
holidays_df = pd.read_csv(RAW_DATA_DIR / 'holidays_events.csv', parse_dates=['date'])
transactions_df = pd.read_csv(RAW_DATA_DIR / 'transactions.csv', parse_dates=['date'])

print('train:', train_df.shape)
print('test:', test_df.shape)
print('stores:', stores_df.shape)
print('oil:', oil_df.shape)
print('holidays:', holidays_df.shape)
print('transactions:', transactions_df.shape)


## Краткая проверка исходных таблиц

Здесь оставлены только те сводки, которые реально нужны для старта экспериментов.
Идея не в том, чтобы повторить весь exploratory analysis, а в том, чтобы быстро
проверить целостность данных перед генерацией артефактов.


In [ ]:
overview = pd.DataFrame([
    {
        'dataset': 'train',
        'rows': len(train_df),
        'date_min': train_df['date'].min(),
        'date_max': train_df['date'].max(),
        'stores': train_df['store_nbr'].nunique(),
        'families': train_df['family'].nunique(),
    },
    {
        'dataset': 'test',
        'rows': len(test_df),
        'date_min': test_df['date'].min(),
        'date_max': test_df['date'].max(),
        'stores': test_df['store_nbr'].nunique(),
        'families': test_df['family'].nunique(),
    },
])
overview


In [ ]:
family_summary = (
    train_df.groupby('family')['sales']
    .agg(['mean', 'median', 'std', 'sum'])
    .sort_values('sum', ascending=False)
    .head(15)
)
family_summary


## Подготовка признаков и таблиц

Ниже используется минимально достаточный конвейер:
- география и тип магазина из `stores.csv`;
- цена нефти из `oil.csv` с приведением к ежедневному календарю;
- иерархия праздников из `holidays_events.csv`;
- количество транзакций только для train-части;
- индикатор `is_payday` как воспроизводимый календарный признак.


In [ ]:
CATEGORY_MAP = {
    'Праздник': [
        'Carnaval', 'Navidad', 'Navidad-1', 'Navidad-2', 'Navidad-3', 'Navidad-4',
        'Navidad+1', 'Dia de la Madre', 'Dia de la Madre-1', 'Dia del Trabajo',
        'Dia de Difuntos', 'Viernes Santo', 'Primer dia del ano', 'Primer dia del ano-1'
    ],
    'Основание': [
        'Fundacion de Cuenca', 'Fundacion de Ibarra', 'Fundacion de Quito', 'Fundacion de Quito-1',
        'Fundacion de Manta', 'Fundacion de Loja', 'Fundacion de Santo Domingo', 'Fundacion de Machala',
        'Fundacion de Esmeraldas', 'Fundacion de Riobamba', 'Fundacion de Ambato', 'Fundacion de Guayaquil',
        'Fundacion de Guayaquil-1', 'Provincializacion de Santo Domingo', 'Provincializacion Santa Elena',
        'Provincializacion de Cotopaxi', 'Provincializacion de Imbabura', 'Cantonizacion de Salinas',
        'Cantonizacion de Libertad', 'Cantonizacion de Riobamba', 'Cantonizacion del Puyo',
        'Cantonizacion de Guaranda', 'Cantonizacion de Latacunga', 'Cantonizacion de El Carmen',
        'Cantonizacion de Cayambe', 'Cantonizacion de Quevedo'
    ],
    'Независимость/История': [
        'Independencia de Guaranda', 'Independencia de Latacunga', 'Independencia de Ambato',
        'Independencia de Cuenca', 'Independencia de Guayaquil', 'Primer Grito de Independencia',
        'Batalla de Pichincha'
    ],
    'Глобальные распродажи': ['Black Friday', 'Cyber Monday'],
    'Футбол': [
        'Mundial de futbol Brasil: Octavos de Final', 'Mundial de futbol Brasil: Cuartos de Final',
        'Mundial de futbol Brasil: Semifinales', 'Inauguracion Mundial de futbol Brasil',
        'Mundial de futbol Brasil: Ecuador-Suiza', 'Mundial de futbol Brasil: Ecuador-Honduras',
        'Mundial de futbol Brasil: Ecuador-Francia', 'Mundial de futbol Brasil: Tercer y cuarto lugar',
        'Mundial de futbol Brasil: Final'
    ],
    'Землетрясение': [
        'Terremoto Manabi', 'Terremoto Manabi+1', 'Terremoto Manabi+2', 'Terremoto Manabi+3',
        'Terremoto Manabi+4', 'Terremoto Manabi+5', 'Terremoto Manabi+6', 'Terremoto Manabi+7',
        'Terremoto Manabi+8', 'Terremoto Manabi+9', 'Terremoto Manabi+10', 'Terremoto Manabi+11',
        'Terremoto Manabi+12', 'Terremoto Manabi+13', 'Terremoto Manabi+14', 'Terremoto Manabi+15',
        'Terremoto Manabi+16', 'Terremoto Manabi+17', 'Terremoto Manabi+18', 'Terremoto Manabi+19',
        'Terremoto Manabi+20', 'Terremoto Manabi+21', 'Terremoto Manabi+22', 'Terremoto Manabi+23',
        'Terremoto Manabi+24', 'Terremoto Manabi+25', 'Terremoto Manabi+26', 'Terremoto Manabi+27',
        'Terremoto Manabi+28', 'Terremoto Manabi+29', 'Terremoto Manabi+30'
    ],
    'Переносы': [
        'Traslado Independencia de Guayaquil', 'Traslado Primer Grito de Independencia',
        'Traslado Batalla de Pichincha', 'Traslado Fundacion de Guayaquil', 'Traslado Fundacion de Quito',
        'Traslado Primer dia del ano', 'Puente Navidad', 'Puente Primer dia del ano',
        'Puente Dia de Difuntos', 'Recupero puente primer dia del ano', 'Recupero Puente Dia de Difuntos',
        'Recupero Puente Navidad', 'Recupero Puente Primer dia del ano', 'Recupero puente Navidad'
    ],
}

def recode_event(event):
    for category, events in CATEGORY_MAP.items():
        if event in events:
            return category
    return 'Другое'


def prepare_oil(oil_raw):
    oil = oil_raw[['date', 'dcoilwtico']].drop_duplicates().sort_values('date').set_index('date')
    oil = oil.asfreq('D')
    oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='time').ffill().bfill()
    return oil.reset_index()


def prepare_holidays(holidays_raw):
    holidays = holidays_raw.copy()
    holidays['agg_description'] = holidays['description'].apply(recode_event)
    holidays = holidays.drop_duplicates()
    return holidays


def build_holiday_views(holidays):
    fields = ['type', 'locale', 'description', 'transferred', 'agg_description']
    national = holidays[holidays['locale'] == 'National'][['date'] + fields].drop_duplicates('date')
    national = national.rename(columns={c: f'{c}_national' for c in fields})

    regional = holidays[holidays['locale'] == 'Regional'][['date', 'locale_name'] + fields].drop_duplicates(['date', 'locale_name'])
    regional = regional.rename(columns={'locale_name': 'state', **{c: f'{c}_regional' for c in fields}})

    local = holidays[holidays['locale'] == 'Local'][['date', 'locale_name'] + fields].drop_duplicates(['date', 'locale_name'])
    local = local.rename(columns={'locale_name': 'city', **{c: f'{c}_local' for c in fields}})
    return national, regional, local


def enrich_frame(df, stores, oil, holidays, transactions=None, is_train=True):
    out = df.copy()
    out = out.merge(stores.rename(columns={'type': 'store_type'}), on='store_nbr', how='left')
    out = out.merge(oil, on='date', how='left')

    national, regional, local = build_holiday_views(holidays)
    out = out.merge(national, on='date', how='left')
    out = out.merge(regional, on=['date', 'state'], how='left')
    out = out.merge(local, on=['date', 'city'], how='left')

    for field in ['type', 'locale', 'description', 'transferred', 'agg_description']:
        out[field] = out.get(f'{field}_national').fillna(out.get(f'{field}_regional')).fillna(out.get(f'{field}_local'))

    out['type'] = out['type'].fillna('Work Day')
    out['locale'] = out['locale'].fillna('None')
    out['description'] = out['description'].fillna('None')
    out['agg_description'] = out['agg_description'].fillna('None')
    out['transferred'] = out['transferred'].fillna(False)
    out['is_holiday'] = (
        out['type'].isin(['Holiday', 'Transfer', 'Bridge']) & (~out['transferred'].astype(bool))
    ).astype('int8')
    out['is_payday'] = out['date'].apply(lambda x: 1 if x.day in [15, x.days_in_month] else 0).astype('int8')

    drop_cols = [col for col in out.columns if col.endswith('_national') or col.endswith('_regional') or col.endswith('_local')]
    out = out.drop(columns=drop_cols, errors='ignore')

    if is_train and transactions is not None:
        out = out.merge(transactions, on=['date', 'store_nbr'], how='left')
        out['transactions'] = out['transactions'].fillna(0.0)

    duplicates = out[['date', 'store_nbr', 'family']].duplicated().sum()
    if duplicates:
        out = out.drop_duplicates(subset=['date', 'store_nbr', 'family'])

    return out.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)


In [ ]:
oil_prepared = prepare_oil(oil_df)
holidays_prepared = prepare_holidays(holidays_df)

train_processed = enrich_frame(
    train_df,
    stores=stores_df,
    oil=oil_prepared,
    holidays=holidays_prepared,
    transactions=transactions_df,
    is_train=True,
)

test_processed = enrich_frame(
    test_df,
    stores=stores_df,
    oil=oil_prepared,
    holidays=holidays_prepared,
    transactions=None,
    is_train=False,
)

train_processed.to_csv(DATASETS_DIR / 'train_processed.csv', index=False)
test_processed.to_csv(DATASETS_DIR / 'test_processed.csv', index=False)

print(train_processed.shape)
print(test_processed.shape)
train_processed.head()


## Артефакты экспериментов

Именно здесь фиксируются сплиты, подмножество `top_pairs` и пары с нулевыми продажами.
После этого все остальные ноутбуки должны работать только с файлами из `artifacts/`.


In [ ]:
def generate_time_splits(
    df,
    n_splits=3,
    horizon_days=16,
    step_days=30,
    date_col='date',
    last_val_end=None,
):
    ordered = df.sort_values(date_col).copy()
    unique_dates = ordered[date_col].drop_duplicates().sort_values().to_numpy()
    last_val_end = pd.to_datetime(unique_dates[-1] if last_val_end is None else last_val_end)
    splits = []
    for k in range(n_splits):
        val_end = last_val_end - pd.Timedelta(days=step_days * k)
        val_start = val_end - pd.Timedelta(days=horizon_days - 1)
        train_end = val_start - pd.Timedelta(days=1)
        train_idx = ordered.index[ordered[date_col] <= train_end]
        val_idx = ordered.index[(ordered[date_col] >= val_start) & (ordered[date_col] <= val_end)]
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue
        splits.append({
            'name': f'split_{k + 1}',
            'train_idx': train_idx.tolist(),
            'val_idx': val_idx.tolist(),
            'train_end': str(pd.to_datetime(train_end).date()),
            'val_start': str(pd.to_datetime(val_start).date()),
            'val_end': str(pd.to_datetime(val_end).date()),
        })
    return splits

splits = generate_time_splits(train_processed, n_splits=3, horizon_days=16, step_days=30)
with open(METADATA_DIR / 'splits.json', 'w', encoding='utf-8') as f:
    json.dump(splits, f, indent=2, ensure_ascii=False)

series_sales = (
    train_processed.groupby(['store_nbr', 'family'])['sales']
    .sum()
    .rename('sales_sum')
    .reset_index()
)
series_sales['sales_share'] = series_sales['sales_sum'] / series_sales['sales_sum'].sum()
quantile_cut = series_sales['sales_sum'].quantile(0.8)
top_pairs_df = series_sales[series_sales['sales_sum'] >= quantile_cut].copy()
zero_sales = (
    train_processed.groupby(['store_nbr', 'family'], as_index=False)['sales']
    .sum()
    .query('sales == 0')
)

top_pairs_df.to_csv(DATASETS_DIR / 'top_pairs.csv', index=False)
zero_sales.to_csv(DATASETS_DIR / 'zero_sales.csv', index=False)

pd.DataFrame(splits), top_pairs_df.head(), zero_sales.head()
